In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("../")

In [3]:
from ease_recommender import *
from npmi_recommender import *

import pickle as p

In [4]:
def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

In [5]:
print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

loading cache data...
building csr matrices...
done


In [6]:
cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

In [7]:
# use two items that you believe are similar to optimize the value of lambda_

a_name = find_match_using_terms(["sgeir", "7xUZ4069zcyBM4Bn10NQ1c"], cat2idx)
a = cat2idx[a_name]

# a = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]

# b = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]
# b = cat2idx[find_match_using_terms(["Bon Iver", "4LEiUm1SRbFMgfqnQTwUbQ"], cat2idx)]
# b = cat2idx[find_match_using_terms(["SOHN"], cat2idx)]

# b_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)
# b_name = find_match_using_terms(["Fleet Foxes"], cat2idx)
b_name = find_match_using_terms(["Highasakite"], cat2idx)
b = cat2idx[b_name]

In [39]:
from tqdm.auto import tqdm

In [44]:
"""
InfoMax Recommender: Learn to recommend items that maximize mutual information
with future user trajectory, inspired by Knowledge Gradient.

Core idea: I(i_{t+1}; s_{t+2:T} | s_{1:t}) approximates information gain,
trained via InfoNCE contrastive objective.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


@dataclass
class Config:
    n_items: int
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 4
    max_seq_len: int = 512
    dropout: float = 0.1
    n_negatives: int = 127
    lr: float = 1e-4
    lambda_kg: float = 0.1


class CausalTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.item_embed = nn.Embedding(cfg.n_items, cfg.d_model)
        self.pos_embed = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        layer = nn.TransformerEncoderLayer(
            cfg.d_model, cfg.n_heads, cfg.d_model * 4,
            dropout=cfg.dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, cfg.n_layers)

    def forward(self, item_ids):
        B, T = item_ids.shape
        pos = torch.arange(T, device=item_ids.device)
        x = self.item_embed(item_ids) + self.pos_embed(pos)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        return self.transformer(x, mask=mask)

    def encode(self, item_ids):
        return self.forward(item_ids)[:, -1, :]


class InfoMaxRecommender(nn.Module):
    """
    Recommender maximizing I(next_item; future | past) via InfoNCE.
    Optional KG-inspired regularization for exploration.
    """
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.encoder = CausalTransformer(cfg)

        # Bilinear critics for InfoNCE
        self.past_critic = nn.Bilinear(cfg.d_model, cfg.d_model, 1)
        self.future_critic = nn.Bilinear(cfg.d_model, cfg.d_model, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))

        # Value head for KG approximation
        self.value_head = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_model),
            nn.GELU(),
            nn.Linear(cfg.d_model, cfg.n_items)
        )

    def get_item_embed(self, item_ids):
        return self.encoder.item_embed(item_ids)

    def critic_score(self, past_repr, item_embed, future_repr):
        """Score (past, item, future) tuples for InfoNCE."""
        if item_embed.dim() == 2:
            item_embed = item_embed.unsqueeze(1)
        B, K, D = item_embed.shape

        past_exp = past_repr.unsqueeze(1).expand(-1, K, -1).reshape(-1, D)
        future_exp = future_repr.unsqueeze(1).expand(-1, K, -1).reshape(-1, D)
        items_flat = item_embed.reshape(-1, D)

        past_score = self.past_critic(past_exp, items_flat).reshape(B, K)
        future_score = self.future_critic(future_exp, items_flat).reshape(B, K)

        a = torch.sigmoid(self.alpha)
        return a * past_score + (1 - a) * future_score

    def info_nce_loss(self, past, present, future):
        """
        InfoNCE: positive = observed item, negatives = random items.
        Maximizes lower bound on I(present; future | past).
        """
        B = past.shape[0]
        device = past.device

        past_repr = self.encoder.encode(past)
        future_repr = self.encoder.encode(future)

        pos_embed = self.get_item_embed(present)
        neg_items = torch.randint(0, self.cfg.n_items, (B, self.cfg.n_negatives), device=device)
        neg_embed = self.get_item_embed(neg_items)

        pos_score = self.critic_score(past_repr, pos_embed, future_repr)
        neg_scores = self.critic_score(past_repr, neg_embed, future_repr)

        logits = torch.cat([pos_score, neg_scores], dim=1)
        labels = torch.zeros(B, dtype=torch.long, device=device)
        return F.cross_entropy(logits, labels)

    def kg_loss(self, past, present, n_samples=5):
        """
        KG-inspired: encourage recommending items that would
        maximally change our value estimates.
        """
        B = past.shape[0]
        D = self.cfg.d_model
        device = past.device

        past_repr = self.encoder.encode(past)
        current_values = self.value_head(past_repr)
        current_best = current_values.max(dim=1).values

        pos_embed = self.get_item_embed(present)
        neg_items = torch.randint(0, self.cfg.n_items, (B, 63), device=device)
        neg_embed = self.get_item_embed(neg_items)
        all_embeds = torch.cat([pos_embed.unsqueeze(1), neg_embed], dim=1)

        # Approximate KG via sampled belief updates
        kg_scores = []
        for c in range(all_embeds.shape[1]):
            future_bests = []
            for _ in range(n_samples):
                noise = torch.randn_like(past_repr) * 0.1
                updated = F.layer_norm(past_repr + all_embeds[:, c] + noise, (D,))
                future_bests.append(self.value_head(updated).max(dim=1).values)
            kg_scores.append(torch.stack(future_bests).mean(0) - current_best)

        kg_scores = torch.stack(kg_scores, dim=1)
        return F.cross_entropy(kg_scores, torch.zeros(B, dtype=torch.long, device=device))

    def loss(self, sequences, lambda_kg=0.1):
        """Combined InfoNCE + KG loss on a batch of sequences."""
        B, T = sequences.shape
        t = T // 2

        past, present, future = sequences[:, :t], sequences[:, t], sequences[:, t+1:]
        if future.shape[1] == 0:
            return torch.tensor(0.0, device=sequences.device)

        info_loss = self.info_nce_loss(past, present, future)
        kg_loss = self.kg_loss(past, present) if lambda_kg > 0 else 0
        return info_loss + lambda_kg * kg_loss

    @torch.no_grad()
    def recommend(self, past, k=100):
        """Recommend top-k items given user history."""
        past_repr = self.encoder.encode(past)
        all_embeds = self.encoder.item_embed.weight

        # Use value head + uncertainty proxy for exploration
        values = self.value_head(past_repr)

        # Simple UCB: value + scaled embedding similarity (proxy for uncertainty)
        similarity = F.cosine_similarity(
            past_repr.unsqueeze(1), all_embeds.unsqueeze(0), dim=-1
        )
        scores = values + 0.1 * (1 - similarity)  # less similar = more uncertain

        return scores.topk(k, dim=-1).indices


def train_info_max_recommender(
    train_sequences: list[list[int]],
    cfg: Config,
    n_epochs: int = 100,
    batch_size: int = 256,
    min_seq_len: int = 10,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    """
    Train InfoMax recommender on user interaction sequences.

    Args:
        train_sequences: List of item ID sequences per user
        cfg: Model configuration
        n_epochs: Training epochs
        batch_size: Batch size
        min_seq_len: Minimum sequence length to use
        device: Training device

    Returns:
        Trained model
    """
    sequences = [torch.tensor(s) for s in train_sequences if len(s) >= min_seq_len]
    if not sequences:
        raise ValueError(f"No sequences with length >= {min_seq_len}")

    model = InfoMaxRecommender(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)

    for epoch in tqdm(range(n_epochs)):
        model.train()
        total_loss, n_batches = 0.0, 0
        indices = torch.randperm(len(sequences))

        for batch_start in tqdm(range(0, len(sequences), batch_size)):
            batch_idx = indices[batch_start:batch_start + batch_size]
            batch_seqs = [sequences[i] for i in batch_idx]

            # Pad to max length in batch
            max_len = max(len(s) for s in batch_seqs)
            padded = torch.zeros(len(batch_seqs), max_len, dtype=torch.long, device=device)
            for i, s in enumerate(batch_seqs):
                padded[i, :len(s)] = s.to(device)

            optimizer.zero_grad()
            loss = model.loss(padded, lambda_kg=cfg.lambda_kg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1
            
            print(total_loss/n_batches)

        scheduler.step()

        if epoch % 10 == 0 or epoch == n_epochs - 1:
            print(f"Epoch {epoch:3d} | Loss: {total_loss/n_batches:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    return model

In [9]:
artist_mat

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 38088211 stored elements and shape (1000000, 295860)>

In [11]:
sample_size = int(10_000)

I = np.random.choice(artist_mat.shape[0], size=sample_size)

In [16]:
X = artist_mat[I].tocoo()

In [19]:
X.row

array([   0,    0,    0, ..., 9999, 9999, 9999],
      shape=(381046,), dtype=int32)

In [20]:
X.col

array([     2,      4,      5, ..., 184584, 194483, 281777],
      shape=(381046,), dtype=int32)

In [24]:
sequences = [[] for _ in range(sample_size)]

for user, item in zip(X.row, X.col):
    sequences[user].append(int(item))

In [31]:
for seq in sequences:
    assert np.diff(seq).min() >= 1

In [49]:
N_USERS, N_ITEMS = X.shape
SEQ_LEN = 128

cfg = Config(n_items=N_ITEMS, d_model=128, n_layers=2, n_heads=4)
model = train_info_max_recommender(sequences, cfg, n_epochs=5, batch_size=64)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/139 [00:00<?, ?it/s]

11.12057876586914
9.838801383972168
9.917452176411947
12.011403799057007
11.16000680923462
9.85111657778422
9.379379681178502
8.870649099349976
8.196562078264025
7.809641361236572
7.38487982749939
7.101501643657684
6.929494656049288
6.738849827221462
6.624640798568725
6.36793977022171
6.144147494260003
5.990952690442403
5.817190320868241
5.6632864594459535
5.507178658530826
5.465858687054027
5.3728609292403515
5.194288884600003
5.178499789237976
5.087683728108039
5.07083902977131
4.988783363785062
4.975222180629599
4.934449668725332
4.883210424453981
4.824452873319387
4.7787077101794155
4.712076597353992
4.631467001778739
4.552679677804311
4.542760468818046
4.551487665427358
4.525532105030158
4.490778875350952
4.4262965888511845
4.4522269283022196
4.392648990764174
4.3729733716357835
4.3631563504536945
4.333110778228097
4.324450949405102
4.285248165329297
4.257879626994231
4.204646570682526
4.188658576385648
4.15165019264588
4.165898676188487
4.1373831011630875
4.16424966508692
4.12951

  0%|          | 0/139 [00:00<?, ?it/s]

1.6432729959487915
2.6407797932624817
2.8582536776860556
2.641981840133667
2.750563144683838
2.9196158250172934
3.147294248853411
2.9479800015687943
3.029183851348029
3.1822478652000425
3.203656727617437
3.300055831670761
3.203280604802645
3.1189261930329457
3.1603458960851034
3.1184997484087944
3.138734025113723
3.144404629866282
3.0987564952749955
3.147329384088516
3.1496121486028037
3.100271533836018
3.111299022384312
3.0379365781943
3.0137602996826174
3.0660506945389967
3.057201385498047
3.0096819358212605
2.983575668828241
2.9250064373016356
2.8970426667121147
2.892579711973667
2.891991297403971
2.9761935963350186
2.946920132637024
2.931063953373167
2.895486706012004
2.8694085823862174
2.8590855048252988
2.8485705316066743
2.8467459504197286
2.8758121842429754
2.8969484761703845
2.9439596967263655
2.9345143530103894
2.9286725054616514
2.8990117970933307
2.897923342883587
2.890894741428142
2.857642288208008
2.845218490151798
2.8316999031947208
2.816896951423501
2.8143862883249917
2

...

2.5084855556488037
2.1584516167640686
2.2545809348424277
2.5899051129817963
2.4192585945129395
2.6726731061935425
2.556952255112784
2.846171960234642
2.9190387858284845
2.826018679141998
2.783256108110601
2.8681448797384896
2.7325705473239603
2.7397804856300354
2.6760321855545044
2.7323445454239845
2.673712400829091
2.629430698023902
2.5686184481570593
2.560938835144043
2.630606378827776
2.647030093453147
2.696670729181041
2.7244449059168496
2.747465524673462
2.776693921822768
2.760788467195299
2.7462515916143144
2.71746786298423
2.658772732814153
2.661632316727792
2.644127892330289
2.6201653679211936
2.6621779045637917
2.6549442069871083
2.6933870630131826
2.663001187749811
2.6543500909679816
2.602453520664802
2.5772492215037346
2.549054478726736
2.5384278311615898
2.535867936389391
2.5290258811278776
2.507917598883311
2.495252286610396
2.511509584619644
2.5096152859429517
2.4944371556749148
2.461244719028473
2.45148294579749
2.436913102865219
2.472613809243688
2.457618949589906
2.463

  0%|          | 0/139 [00:00<?, ?it/s]

2.5340452194213867
2.6079630851745605
2.4043215115865073
2.194721519947052
2.2239795207977293
2.1110565066337585
2.08113089629582
2.097071185708046
1.9924966361787584
2.139702332019806
2.1670587821440264
2.1176378627618155
2.105782380470863
2.027964915548052
2.126354217529297
2.1157625392079353
2.1428737430011524
2.2045740617646112
2.148198811631454
2.147558242082596
2.1595015014920915
2.207287707112052
2.2767866020617276
2.3763446857531867
2.3866416025161743
2.358839598985819
2.3791514016963817
2.3511236352579936
2.3152711761408837
2.3080124974250795
2.31486383176619
2.301982816308737
2.331653179544391
2.379802833585178
2.410802047593253
2.4123256968127356
2.4275547845943555
2.4535423925048425
2.447581398181426
2.4475777477025984
2.404415630712742
2.4174027953829085
2.510049126869024
2.4930136149579827
2.503689850701226
2.5508269019748853
2.530623240673796
2.512246958911419
2.5006619667520327
2.467101399898529
2.446236121888254
2.4752151255424204
2.4539471054976842
2.449839898833522
2

  0%|          | 0/139 [00:00<?, ?it/s]

4.061498641967773
2.323644280433655
2.204692324002584
2.1604590713977814
2.1118988037109374
2.1683181126912436
2.2563702378954207
2.2837940454483032
2.507316748301188
2.375868093967438
2.3988327221436934
2.5460144778092704
2.5418755549650927
2.684431459222521
2.6594149351119993
2.5376785062253475
2.4815296670969795
2.43943609462844
2.5551718191096655
2.5225678414106367
2.6105660455567494
2.6746164858341217
2.6889276374941287
2.820913481215636
2.7848745465278624
2.861294005925839
2.938683044027399
2.942514012966837
2.932392656803131
2.876610463857651
2.8694022028676924
2.824033485725522
2.8116827896147063
2.8168364710667553
2.78935991866248
2.750092471639315
2.7234884806581445
2.709224533093603
2.6637397087537327
2.727319982647896
2.7242168711452948
2.7065713632674444
2.6989363570545994
2.6641524081880394
2.6569318638907538
2.6158278675183007
2.606488158094122
2.555960334216555
2.5513887910210356
2.5287644189596175
2.538279855368184
2.559022875359425
2.552352193953856
2.5760116571629488

In [50]:
"done"

'done'

In [83]:
# Test recommendation
model.eval()
test_seq = torch.tensor([b, a], device=next(model.parameters()).device).unsqueeze(0)
recs = model.recommend(test_seq, k=10)
print(f"Recommendations: {recs[0].tolist()}")

Recommendations: [166234, 275188, 258811, 283866, 249320, 209928, 134050, 42565, 151882, 242773]


In [84]:
for rec in recs[0].tolist():
    print(idx2cat[rec])

Chelsea Moon (spotify:artist:4oY8BnnDy5wv6P58kOieeV)
Just The Boyz (spotify:artist:7bTd25PCpnNEtOKtxAcsEx)
Low Pressure (spotify:artist:1ucqmuQ9ZU7530mjCC3I1Q)
Johnny Diesel & The Injectors (spotify:artist:0Nlg1HCtb8dpeVpxQojDiq)
Saad-Eddine El Andaloussi (spotify:artist:5KTtmbvxoyP6QyXdTNMAiX)
Discover Worship (spotify:artist:0Nca7W3ogv4nv9jWbv92rQ)
Hades (spotify:artist:3MblUyzg5WUKZQhWGpyk5B)
T and Sugah (spotify:artist:6jsS2mOTAxVrlSUWiPLXpH)
Lyktum (spotify:artist:42qQU30gkiyVstokBfGiex)
Captain DaFeira (spotify:artist:2IJF6IoX258ZLRWamLnull)


In [69]:
idx2cat[a]

'Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)'

In [58]:
b

5971

In [57]:
a

9060

In [54]:
test_seq.shape

torch.Size([2])

In [ ]:
# Test recommendation
model.eval()
test_seq = torch.tensor([sequences[0][:25]], device=next(model.parameters()).device)
recs = model.recommend(test_seq, k=10)
print(f"\nTest sequence (last 5): {sequences[0][20:25]}")
print(f"Recommendations: {recs[0].tolist()}")

In [8]:
lambda_ = optimize_lambda_using_a_to_b_matching(artist_mat, a, b, fast_approximation=True)

lambda_: 1000000
error: 339.0
lambda_: 100000.0
error: 157.0
lambda_: 10000.0
error: 38.0
lambda_: 1000.0
error: 5.0
lambda_: 100.0
error: 3.0
lambda_: 10.0
error: 3.5


In [10]:
# lambda_ = optimize_lambda_using_a_to_b_matching(artist_mat, a, b, fast_approximation=False)

In [11]:
# check final error for EASE
a_to_b_error_metric(artist_mat, a, b, lambda_)

lambda_: 100
error: 3.0


3.0

In [12]:
temp = 1

In [13]:
# check error for NPMI in comparison
a_to_b_error_metric_npmi(artist_mat, a, b, temp)

temp: 1
error: 903.0


903.0

In [14]:
top_k = 20

In [15]:
# using EASE

similarity_scores = calculate_ease_for_item_cg(artist_mat, a, lambda_)
top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

['Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)',
 'PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)',
 'The Acid (spotify:artist:0bRtSoJSpQdnbB3dWrWprR)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)',
 'The Careful Ones (spotify:artist:1DdAoWvETBUklcJCOISZx1)',
 'Liza Anne (spotify:artist:426VSUSxx9puUYFgp7l7EQ)',
 'Snakadaktal (spotify:artist:0SdEkx5Ai2gl0W7pnhlsfy)',
 'Vök (spotify:artist:7oDTyDfeA2JE2jUZztkBj8)',
 'Seoul (spotify:artist:3e69LorE0YSsEaYY6x9XuG)',
 'Only Real (spotify:artist:5cyHu7tidauRJ9UawaPwG5)',
 'Big Scary (spotify:artist:4mLYW48jy9Pwv6KpT74Evf)',
 'SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)',
 'San Fermin (spotify:artist:7fSnislKgW9Mz0YIqWQmGt)',
 'Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)',
 'Caroline Smith (spotify:artist:47blM5Op3BJODxUJImwdYE)',
 'Neulore (spotify:artist:6SLbaDa56f1CZlUNuF0gk3)',
 'Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)',
 'Ásgeir Trausti (spotif

In [16]:
# using EASE

similarity_scores = calculate_ease_for_item_cg(artist_mat, b, lambda_)
top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

['Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Alice Boman (spotify:artist:3WiytRnvoL0kT3oAGl9TCt)',
 'Majical Cloudz (spotify:artist:4BEYBN6NCPrFk3sOLMTby3)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Rubblebucket (spotify:artist:6xriZDSK3wPXhOoZXr9fzF)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'Max Jury (spotify:artist:3MuPVbFDynbq9zRTAqjRxi)',
 'SOAK (spotify:artist:4PLsMEk2DCRVlVL2a9aZAv)',
 'Amason (spotify:artist:4cJKxS7uOPhwb5UQ70sYpN)',
 'Joseph (spotify:artist:5Wfvw7rDz7HA6gE2z6QhqO)',
 'MUNA (spotify:artist:6xdRb2GypJ7DqnWAI2mHGn)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'The Dø (spotify:artist:2mcNCn1qbZUQ3J9KHapUxj)',
 'Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)',
 'Matrimony (spotify:a

In [17]:
# using normalized pointwise mutual information

similarity_scores = npmi_batch(artist_mat, a, temp)

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

['Low Volts (spotify:artist:3PxUwSSsVaW0XyBiRJF2oS)',
 'Jennifer Sullivan (spotify:artist:6Pw9rPhhbzGfWM5nomrH3A)',
 'Shields of Faith (spotify:artist:2hr9UmcGXibjVZe8Binikn)',
 'Blackpocket (spotify:artist:03HmbNfPUCurmGnU4NNUhF)',
 'Gluteus Maximus (spotify:artist:0q0nbjiNlgkzLclUx2m79K)',
 'Ásgeir Trausti (spotify:artist:7fNWySjsDn74LCawyJ27EQ)',
 'Auden (spotify:artist:5HMe9gHJrucpCmLNd2iPSF)',
 'Josienne Clarke and Ben Walker (spotify:artist:3Vur5nBUAcGQjXZyk32WV1)',
 'Cy Jack|Duncan Aran (spotify:artist:53ZZVIzwhRIE7rDMVPaqFw)',
 'Adurn (spotify:artist:4hDzPWJ7vNSD2wCEF49w8r)',
 'James O-L (spotify:artist:1zhN3uwQgyxwLVE2piO0AM)',
 'Labrador Labratories (spotify:artist:7ExeTdXwDt70bNROpS4Gr5)',
 'Macoubre (spotify:artist:1Bsx51kT7wbNEJll4wtcCi)',
 'Kristopher James (spotify:artist:13o5y7dYYnsyKpbq162GIf)',
 'The Project Club (spotify:artist:2A7GpAZpK9GLtYvgTHhuCm)',
 'John Gurney (spotify:artist:689fsusZuLMRcbgu9yImOu)',
 'The Violet Jive (spotify:artist:0g0XgdmP4KpcTUiM5xYA2e)',